# Render Duo Video: Orang Indonesia Pertama & Sejarah Majapahit di Google Colab (T4 GPU)

Notebook render otomatis untuk **2 video** secara berurutan tanpa menu opsional:
1. **Orang Indonesia Pertama** (*Style Stickman*):
   - Resolusi: 1920x1080 @ 30fps | Durasi: 307,20 detik (9.216 frame, ~5,12 menit) | 95 scene visual
   - Header Kiri: `OTAK ATIK` | Subtitle: Rounded stickman badge interaktif
2. **Sejarah Majapahit** (*Style Dokumenter Sejarah*):
   - Resolusi: 1920x1080 @ 30fps | Durasi: 631,72 detik (18.952 frame, ~10,53 menit) | 148 scene visual
   - Header Kiri: `SEJARAH MAJAPAHIT` | Header Kanan: `ZEELACUNA` | Subtitle: Arsip serif elegan

---
### Langkah Menjalankan:
1. Pastikan runtime Colab memakai **T4 GPU** (*Runtime -> Change runtime type -> T4 GPU*).
2. Jalankan **Sel 1 (Mount Google Drive)** agar hasil video otomatis tersimpan permanen di Google Drive (`remotion-out/`).
3. Jalankan **Sel 2 (Setup Environment)** untuk instalasi Chromium Linux, Node.js 20, dan clone repo.
4. Jalankan **Sel 3 (Render Otomatis 2 Video)** — kedua video langsung dirender berturut-turut tanpa prompt tambahan.
5. Jalankan **Sel 4 (Download Video)** jika ingin mengunduh langsung ke komputer/laptop.


In [ ]:
# Sel 1: Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
out_dir = '/content/drive/MyDrive/remotion-out'
os.makedirs(out_dir, exist_ok=True)
print('Google Drive terhubung! Output video akan disimpan di:', out_dir)


In [ ]:
%%shell
# Sel 2: Setup Environment, Dependensi Chromium & Clone Repo
set -e

echo "=== 1. Cek GPU ==="
if command -v nvidia-smi >/dev/null 2>&1 && nvidia-smi >/dev/null 2>&1; then
  nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
else
  echo "WARNING: GPU NVIDIA tidak aktif! Disarankan pilih Runtime -> Change runtime type -> T4 GPU."
fi

echo "=== 2. Install dependensi Chromium & Font Linux ==="
DEBIAN_FRONTEND=noninteractive apt-get update -qq
DEBIAN_FRONTEND=noninteractive apt-get install -y -qq \
  libnss3 libnspr4 libdbus-1-3 libatk1.0-0 libatk-bridge2.0-0 libcups2 \
  libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 \
  libxrandr2 libgbm1 libpango-1.0-0 libcairo2 libxshmfence1 \
  libvulkan1 fonts-liberation
(DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2t64 || DEBIAN_FRONTEND=noninteractive apt-get install -y -qq libasound2)

echo "=== 3. Install Node.js 20 ==="
NODE_MAJOR=$(node -v 2>/dev/null | sed 's/v\([0-9]*\).*/\1/' || echo 0)
if [ "$NODE_MAJOR" -lt 20 ]; then
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
  DEBIAN_FRONTEND=noninteractive apt-get install -y -qq nodejs
fi
node -v && npm -v

echo "=== 4. Clone / Update Repo & npm ci ==="
if [ -d /content/vidio/.git ]; then
  cd /content/vidio && git pull --ff-only
else
  rm -rf /content/vidio
  git clone --depth 1 https://github.com/giestar-id/remotion-vidio.git /content/vidio
fi
cd /content/vidio && npm ci

echo "Setup selesai! Siap merender OrangIndoPertama dan SejarahMajapahit."


In [ ]:
# Sel 3: Render Otomatis 2 Video (Orang Indonesia Pertama & Sejarah Majapahit)
import importlib, os, subprocess, sys, time

if os.path.isdir('/content/vidio/.git'):
    subprocess.run(['git', '-C', '/content/vidio', 'pull', '--ff-only'], check=False)

out_dir = '/content/drive/MyDrive/remotion-out' if os.path.isdir('/content/drive/MyDrive') else '/content/remotion-out'
os.makedirs(out_dir, exist_ok=True)

if '/content/vidio/scripts' not in sys.path:
    sys.path.insert(0, '/content/vidio/scripts')

import colab_render
importlib.reload(colab_render)

start_all = time.time()

# =====================================================================
# 1. RENDER ORANG INDONESIA PERTAMA (~5,12 Menit / 9.216 Frame)
# =====================================================================
target_orang_indo = os.path.join(out_dir, 'orang-indo-pertama-1080p.mp4')
print('=== [1/2] MEMULAI RENDER: OrangIndoPertama ===')
colab_render.run_render(
    composition='OrangIndoPertama',
    out_path=target_orang_indo,
    cwd='/content/vidio',
)

# =====================================================================
# 2. RENDER SEJARAH MAJAPAHIT (~10,53 Menit / 18.952 Frame)
# =====================================================================
target_majapahit = os.path.join(out_dir, 'sejarah-majapahit-1080p.mp4')
print('\n=== [2/2] MEMULAI RENDER: SejarahMajapahit ===')
colab_render.run_render(
    composition='SejarahMajapahit',
    out_path=target_majapahit,
    cwd='/content/vidio',
)

total_min = (time.time() - start_all) / 60.0
print(f'\n[SELESAI] Seluruh 2 video berhasil dirender dalam {total_min:.2f} menit!')
print('Lokasi penyimpanan:', out_dir)


In [ ]:
# Sel 4: Cek Hasil Render & Download ke Komputer Lokal
import os
from google.colab import files

out_dir = '/content/drive/MyDrive/remotion-out' if os.path.isdir('/content/drive/MyDrive') else '/content/remotion-out'
video_files = [
    os.path.join(out_dir, 'orang-indo-pertama-1080p.mp4'),
    os.path.join(out_dir, 'sejarah-majapahit-1080p.mp4')
]

print('Daftar file video hasil render:')
for v in video_files:
    if os.path.exists(v):
        size_mb = os.path.getsize(v) / (1024 * 1024)
        print(f'  - {os.path.basename(v)}: {size_mb:.2f} MB')
    else:
        print(f'  - {os.path.basename(v)}: Belum selesai / tidak ditemukan')

# Uncomment baris di bawah ini jika ingin men-download otomatis ke laptop:
# for v in video_files:
#     if os.path.exists(v):
#         files.download(v)
